<a href="https://colab.research.google.com/github/Ayustar/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayustar/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (page) per report date.

For my Lane 2 analysis, I will aggregate daily performance
to page-level signals over a 30-day window. The unit of
analysis is one content item with its trailing-30-day
aggregated signals.

Time window: 2026-03 (mid-panel month, not the final month).
I will use this month to develop features and verify my contract.
The final month (June 2026, in _sample) is sealed as test data.

In [19]:
import os
from google.colab import userdata
import duckdb

# Get token from Colab secrets
token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = token
print("Token loaded")

# Connect to DuckDB
conn = duckdb.connect()

# Install and load extensions
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

# Set the token as an environment variable that DuckDB's httpfs will pick up
# DuckDB httpfs uses standard HTTP headers - we need to set it differently
import subprocess
os.environ['HF_TOKEN'] = token

# Alternative: use fsspec with huggingface filesystem
try:
    import fsspec
    from huggingface_hub import HfFileSystem

    # Create Hugging Face filesystem
    fs = HfFileSystem(token=token)

    # List files to verify access
    files = fs.glob("datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet")
    print(f"Found {len(files)} parquet files")
    for f in files[:3]:
        print(f"  {f}")

except ImportError:
    print("huggingface_hub not installed, trying direct approach...")

# Direct approach using huggingface_hub to download and read
from huggingface_hub import hf_hub_download
import pandas as pd

# Download one file to test
repo_id = "FlyRank/internship-warehouse"
filename = "fact_content_daily_performance/month=2026-03/data_0.parquet"

try:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset",
        token=token
    )
    print(f"Downloaded to: {local_path}")

    # Read with DuckDB
    result = conn.execute(f"""
        SELECT
            COUNT(*) as total_rows,
            COUNT(DISTINCT content_id) as unique_pages,
            MIN(report_date) as min_date,
            MAX(report_date) as max_date
        FROM read_parquet('{local_path}')
    """).fetchdf()

    print("\nGrain verification for month=2026-03:")
    print(result)

except Exception as e:
    print(f"Error: {e}")

Token loaded
Found 1 parquet files
  datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet
Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
Error: Binder Error: Referenced column "content_id" not found in FROM clause!
Candidate bindings: "content_hash_id", "client_hash_id", "month", "scroll_events", "sessions_paid"

LINE 4:             COUNT(DISTINCT content_id) as unique_pages,
                                   ^


In [20]:
# Check the actual columns in the file
schema = conn.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{local_path}')
""").fetchdf()

print("Available columns:")
print(schema[['column_name', 'column_type']].to_string())


Available columns:
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGI

In [21]:
# Grain verification for month=2026-03
result = conn.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT content_hash_id) as unique_pages,
        COUNT(DISTINCT client_hash_id) as unique_clients,
        MIN(report_date) as min_date,
        MAX(report_date) as max_date,
        COUNT(DISTINCT report_date) as unique_dates
    FROM read_parquet('{local_path}')
""").fetchdf()

print("Grain verification for month=2026-03:")
print(result)

# Prove the grain: report_date × content_hash_id should be unique
grain_check = conn.execute(f"""
    SELECT report_date, content_hash_id, COUNT(*) as c
    FROM read_parquet('{local_path}')
    GROUP BY report_date, content_hash_id
    HAVING c > 1
    LIMIT 5
""").fetchdf()

print(f"\nGrain check (should be empty = unique grain holds):")
print(grain_check)
print(f"Duplicate grain rows found: {len(grain_check)}")

# Availability check: filter with IS TRUE
available_check = conn.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_available_rows,
        COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) as has_gsc_clients,
        COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) as has_ga4_clients
    FROM read_parquet('{local_path}')
""").fetchdf()

print(f"\nAvailability check:")
print(available_check)

Grain verification for month=2026-03:
   total_rows  unique_pages  unique_clients   min_date   max_date  \
0     9841378        331437              55 2026-03-01 2026-03-31   

   unique_dates  
0            31  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Grain check (should be empty = unique grain holds):
Empty DataFrame
Columns: [report_date, content_hash_id, c]
Index: []
Duplicate grain rows found: 0

Availability check:
   total_rows  gsc_available_rows  ga4_available_rows  has_gsc_clients  \
0     9841378             3611061              413966          9841378   

   has_ga4_clients  
0          6822637  


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature columns
Knowable at the decision moment because they are historical counts and averages already observed in the trailing window:

- `gsc_impressions` — total search impressions already counted
- `gsc_clicks` — total search clicks already observed
- `gsc_avg_position` — average search position already measured
- `ga4_pageviews` — total pageviews already recorded
- `ga4_sessions` — total sessions already counted
- `ga4_users` — total users already observed
- `ga4_engaged_sessions` — engaged sessions already measured
- `ga4_total_engagement_sec` — total engagement time already recorded
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid` — traffic source breakdowns already observed
- `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` — AI traffic breakdown already measured
- `scroll_events` — scroll engagement already counted

### Label / proxy
The thing I predict, never a feature:

- `is_declining` — to be built from a future window (e.g., did average position drop or did impressions fall over the next 30 days?). Not present in raw data; I construct it from future observations.

### Context
For joining, grouping, splitting — never for the model to learn from:

- `report_date` — the date of the observation row
- `client_hash_id` — pseudonymized client identifier
- `content_hash_id` — pseudonymized content identifier
- `month` — partition key for the warehouse table

### Excluded (with why)

- `gsc_sum_position` — excluded because sum of positions is not meaningful. Position is a rank (1 = best), and summing ranks across different impression counts produces a number that scales with volume, not quality. I use `gsc_avg_position` instead.

- `client_has_gsc`, `client_has_ga4` — excluded because these are client-level flags, not page-level signals. They do not vary between pages in the same client, so they cannot help predict which specific page is declining.

- `gsc_data_available`, `ga4_data_available` — excluded because these are data quality filters, not predictive signals. Using them as features would teach the model a tautology ("no data means zero impressions") rather than a genuine predictive pattern about page decline.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grain check: prove one row = one page on one day
grain_check = conn.execute(f"""
    SELECT report_date, content_hash_id, COUNT(*) as c
    FROM read_parquet('{local_path}')
    GROUP BY report_date, content_hash_id
    HAVING c > 1
    LIMIT 5
""").fetchdf()

print(f"Grain check: {len(grain_check)} duplicate rows found")
print("(Should be 0 for unique grain to hold)")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check: 0 duplicate rows found
(Should be 0 for unique grain to hold)
Empty DataFrame
Columns: [report_date, content_hash_id, c]
Index: []


In [24]:
# Count and date span
result = conn.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT content_hash_id) as unique_pages,
        COUNT(DISTINCT client_hash_id) as unique_clients,
        MIN(report_date) as min_date,
        MAX(report_date) as max_date,
        COUNT(DISTINCT report_date) as unique_dates
    FROM read_parquet('{local_path}')
""").fetchdf()

print("Row count and date span for month=2026-03:")
print(result)

Row count and date span for month=2026-03:
   total_rows  unique_pages  unique_clients   min_date   max_date  \
0     9841378        331437              55 2026-03-01 2026-03-31   

   unique_dates  
0            31  


In [25]:
# Availability check with IS TRUE
availability = conn.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_available,
        ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 1) as gsc_pct,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_available,
        ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 1) as ga4_pct
    FROM read_parquet('{local_path}')
""").fetchdf()

print("Availability check (filtering with IS TRUE):")
print(availability)

Availability check (filtering with IS TRUE):
   total_rows  gsc_available  gsc_pct  ga4_available  ga4_pct
0     9841378        3611061     36.7         413966      4.2


In [26]:
# Build 5-feature frame for pages with GSC data available
features = conn.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) as avg_position_30d,
        SUM(gsc_impressions) as total_impressions_30d,
        SUM(gsc_clicks) as total_clicks_30d,
        SUM(ga4_engaged_sessions) as total_engaged_sessions_30d,
        SUM(sessions_organic) as total_organic_sessions_30d
    FROM read_parquet('{local_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    LIMIT 10
""").fetchdf()

print("5-feature frame (first 10 rows):")
print(features)

5-feature frame (first 10 rows):
            content_hash_id           client_hash_id  avg_position_30d  \
0  content_05597932fe4da067  client_73cda7b4e4f265ea          2.714744   
1  content_7a105f548d9c6916  client_73cda7b4e4f265ea          7.209549   
2  content_905aa32a0230694e  client_73cda7b4e4f265ea          6.481453   
3  content_a3ea9792f793ec72  client_73cda7b4e4f265ea          2.987198   
4  content_36c36abc7650d7af  client_73cda7b4e4f265ea          6.724039   
5  content_05434271b257bb68  client_73cda7b4e4f265ea          6.320337   
6  content_22610b0934f8825e  client_73cda7b4e4f265ea         12.791667   
7  content_712c365258cee05c  client_73cda7b4e4f265ea          4.950311   
8  content_5d412fba6e1a2582  client_73cda7b4e4f265ea          9.445635   
9  content_1f380a642aed423b  client_73cda7b4e4f265ea          6.014516   

   total_impressions_30d  total_clicks_30d  total_engaged_sessions_30d  \
0                   57.0               0.0                         0.0   
1   

Feature availability explanations:

- `avg_position_30d`: knowable at the decision moment because it is the
  average search position already observed over the trailing 30-day window.

- `total_impressions_30d`: knowable at the decision moment because it is
  the sum of search impressions already counted in the historical window.

- `total_clicks_30d`: knowable at the decision moment because it is the
  sum of search clicks already observed in the trailing window.

- `total_engaged_sessions_30d`: knowable at the decision moment because
  it is the sum of engaged sessions already recorded in the historical period.

- `total_organic_sessions_30d`: knowable at the decision moment because
  it is the sum of organic traffic sessions already measured in the trailing window.

In [27]:
# HONEST feature frame (no leakage)
honest = conn.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) as avg_position_30d,
        SUM(gsc_impressions) as total_impressions_30d,
        SUM(gsc_clicks) as total_clicks_30d,
        SUM(ga4_engaged_sessions) as total_engaged_sessions_30d,
        SUM(sessions_organic) as total_organic_sessions_30d,
        CASE
            WHEN AVG(gsc_avg_position) > 15 THEN 1
            ELSE 0
        END as is_declining_label
    FROM read_parquet('{local_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    LIMIT 10
""").fetchdf()

print("HONEST feature frame with proxy label:")
print(honest)

# LEAKY version: adding a column derived from the same logic as the label
leaky = conn.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) as avg_position_30d,
        SUM(gsc_impressions) as total_impressions_30d,
        SUM(gsc_clicks) as total_clicks_30d,
        SUM(ga4_engaged_sessions) as total_engaged_sessions_30d,
        SUM(sessions_organic) as total_organic_sessions_30d,
        CASE
            WHEN AVG(gsc_avg_position) > 15 THEN 1
            ELSE 0
        END as is_declining_label,
        -- LEAKAGE: same threshold as label, would make model appear perfect
        CASE
            WHEN AVG(gsc_avg_position) > 15 THEN 1
            ELSE 0
        END as position_flag_leaky
    FROM read_parquet('{local_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    LIMIT 10
""").fetchdf()

print("\nLEAKY feature frame (deliberate trap):")
print(leaky)

HONEST feature frame with proxy label:
            content_hash_id           client_hash_id  avg_position_30d  \
0  content_ac8663da7484669a  client_62f4a7e64f5e0096          4.909314   
1  content_39d7361b4945d504  client_62f4a7e64f5e0096          4.074107   
2  content_d49a012dcb924e31  client_62f4a7e64f5e0096          5.177774   
3  content_614baf2af4330bd7  client_62f4a7e64f5e0096          4.685335   
4  content_225dc9235023be5f  client_62f4a7e64f5e0096         17.148172   
5  content_7dbc094b799e05a4  client_62f4a7e64f5e0096          5.956862   
6  content_40b10da45f4c1cb5  client_62f4a7e64f5e0096         12.977513   
7  content_df22bda1218f13ff  client_62f4a7e64f5e0096          3.066796   
8  content_cfad137c1b04251b  client_62f4a7e64f5e0096         10.127319   
9  content_9e7c70abfbae371e  client_62f4a7e64f5e0096          5.467633   

   total_impressions_30d  total_clicks_30d  total_engaged_sessions_30d  \
0                   34.0               0.0                         NaN  

THE TRAP: I deliberately added `position_flag_leaky` which uses the exact
same threshold as my proxy label (`is_declining_label`). If I used this as
a feature, the model would score near-perfectly because it is essentially
seeing the answer in advance. This is data leakage — using information that
would not be available at the real prediction moment. I removed this column
and kept only features that are truly knowable before prediction: historical
counts and averages from the trailing window.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One named limitation of this slice:

Only 36.7% of rows in the March 2026 partition have actual GSC data
(gsc_data_available IS TRUE). The remaining ~63% are zero-filled
placeholders from pages with no measurable search presence. This means
my analysis is limited to pages that already have search activity — I
cannot make recommendations for pages with no GSC history using these
signals alone. I also cannot distinguish between "page has no impressions"
and "page has no GSC connection" without the data quality flags, which
I have excluded from features.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.